# Pipeline E2E: Limpieza de datos completa en Polars

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/17_polars/code/02_pipeline_e2e.ipynb)

Este notebook acompaña `04_pipeline_e2e.md` y construye un pipeline de limpieza de datos
**completo y realista** usando exclusivamente Polars en modo lazy.

**Objetivo:** Ver cómo las piezas individuales — expresiones, contextos, tipos, evaluación lazy —
se combinan en un flujo de trabajo real, y cómo el optimizador de Polars reescribe nuestro
plan para ejecutarlo de la manera más eficiente posible.

**Patrón que seguimos:** construir → inspeccionar (`.explain()`) → entender.

```
ventas.csv (sucio, 12+ cols)    productos.csv
        │                           │
        ▼ scan_csv                  ▼ scan_csv
        │                           │
        ▼ Validación de esquema     │
        │ (cast tipos)              │
        ▼ Filtrar filas inválidas   │
        │ (precios negativos,       │
        │  nulls en campos clave)   │
        ▼ Limpiar strings           │
        │ (trim, lowercase)         │
        ▼ Parsear fechas            │
        │ (str → Date)              │
        ▼ Join ◄────────────────────┘
        │ (enriquecer con categoría)
        ▼ Agregar
        │ (por categoría + mes)
        ▼ .explain()
        │ (inspeccionar plan optimizado)
        ▼ .collect()
        │ (EJECUTAR todo)
        ▼ write_parquet
        │
        resultado.parquet (limpio)
```

In [1]:
%pip install polars==1.27.0 numpy

Defaulting to user installation because normal site-packages is not writeable


Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl
import numpy as np
import os
import tempfile
import time

print(f"Polars version: {pl.__version__}")
print(f"NumPy version:  {np.__version__}")

Polars version: 1.27.0
NumPy version:  1.26.4


---

## §1: Generar datos sintéticos sucios

Antes de construir el pipeline, necesitamos datos realistas con los problemas típicos
que encontramos en el mundo real:

| Problema | Ejemplo | Frecuencia |
|----------|---------|------------|
| Espacios extra en strings | `" alice "`, `"  Bob"` | ~30% de filas |
| Mayúsculas inconsistentes | `"ALICE"`, `"alice"`, `"Alice "` | ~50% de filas |
| Precios negativos | `-15.50` (error de captura) | ~2% de filas |
| Nulls dispersos | `None` en precio, cantidad, etc. | ~5% por columna |
| Fechas como strings | `"2025-03-15"` en vez de `Date` | 100% |
| Columnas innecesarias | `padding_01` ... `padding_06` | 6 columnas basura |
| Filas duplicadas | Registros repetidos exactos | ~1% |

Generamos **dos archivos CSV**:
1. `ventas_sucias.csv` — ~50,000 filas con 14 columnas (8 útiles + 6 de relleno)
2. `productos.csv` — ~200 filas de catálogo de productos

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica el escenario de datos sucios
> que motiva este pipeline.

In [3]:
# ── Semilla fija para reproducibilidad ──
np.random.seed(42)

N_VENTAS = 50_000
N_PRODUCTOS = 200

# ── Directorio temporal para los CSVs ──
tmpdir = tempfile.mkdtemp(prefix="polars_pipeline_")
path_ventas = os.path.join(tmpdir, "ventas_sucias.csv")
path_productos = os.path.join(tmpdir, "productos.csv")
path_resultado = os.path.join(tmpdir, "resumen_ventas.parquet")

print(f"Directorio temporal: {tmpdir}")

Directorio temporal: /tmp/polars_pipeline_yqkw4qyg


In [4]:
# ── Generar productos.csv ──
categorias = ["Electrónica", "Ropa", "Hogar", "Deportes", "Alimentos"]
subcategorias = {
    "Electrónica": ["Smartphones", "Laptops", "Tablets", "Audio", "Accesorios"],
    "Ropa":        ["Camisas", "Pantalones", "Zapatos", "Accesorios", "Deportiva"],
    "Hogar":       ["Muebles", "Cocina", "Decoración", "Limpieza", "Jardín"],
    "Deportes":    ["Fitness", "Running", "Natación", "Ciclismo", "Camping"],
    "Alimentos":   ["Frescos", "Enlatados", "Bebidas", "Snacks", "Congelados"],
}
proveedores = ["ProvA", "ProvB", "ProvC", "ProvD", "ProvE", "ProvF", "ProvG", "ProvH"]

prod_ids = list(range(1, N_PRODUCTOS + 1))
prod_cats = np.random.choice(categorias, N_PRODUCTOS)
prod_subcats = [np.random.choice(subcategorias[c]) for c in prod_cats]
prod_provs = np.random.choice(proveedores, N_PRODUCTOS)

df_productos = pl.DataFrame({
    "producto_id": prod_ids,
    "categoria":   prod_cats.tolist(),
    "subcategoria": prod_subcats,
    "proveedor":   prod_provs.tolist(),
})

df_productos.write_csv(path_productos)
print(f"productos.csv: {df_productos.shape}")
df_productos.head(5)

productos.csv: (200, 4)


producto_id,categoria,subcategoria,proveedor
i64,str,str,str
1,"""Deportes""","""Running""","""ProvH"""
2,"""Alimentos""","""Bebidas""","""ProvC"""
3,"""Hogar""","""Muebles""","""ProvE"""
4,"""Alimentos""","""Frescos""","""ProvF"""
5,"""Alimentos""","""Snacks""","""ProvA"""


In [5]:
# ── Generar ventas_sucias.csv ──

# Nombres de clientes con suciedad intencional: espacios extra, mayúsculas mixtas
nombres_base = ["alice", "bob", "carlos", "diana", "elena",
                "fernando", "gabriela", "hugo", "irene", "jaime",
                "karla", "luis", "maria", "nelson", "olivia"]

def ensuciar_nombre(nombre):
    """Introduce suciedad realista en un string: espacios extra, mayúsculas mixtas."""
    variantes = [
        nombre,                         # limpio:    "alice"
        nombre.upper(),                 # todo may:  "ALICE"
        nombre.title(),                 # title:     "Alice"
        f"  {nombre}",                  # espacio antes
        f"{nombre}  ",                  # espacio después
        f" {nombre.upper()} ",          # combinado
        f"  {nombre.title()}  ",        # doble espacio + title
    ]
    return variantes[np.random.randint(len(variantes))]

# Nombres de vendedores (también sucios)
vendedores_base = ["juan pérez", "ana garcía", "pedro lópez",
                   "sofía martínez", "miguel ángel ruiz"]

# Generar columnas principales
clientes = [ensuciar_nombre(np.random.choice(nombres_base)) for _ in range(N_VENTAS)]
vendedores = [ensuciar_nombre(np.random.choice(vendedores_base)) for _ in range(N_VENTAS)]

# Producto_id: referencia al catálogo
producto_ids = np.random.randint(1, N_PRODUCTOS + 1, N_VENTAS)

# Precios: mayormente positivos, ~2% negativos (errores de captura)
precios = np.round(np.random.exponential(scale=150, size=N_VENTAS), 2)
mask_negativo = np.random.random(N_VENTAS) < 0.02
precios[mask_negativo] = -precios[mask_negativo]  # invertir signo

# Cantidades: 1-20, algunos negativos
cantidades = np.random.randint(1, 21, N_VENTAS)
mask_cant_neg = np.random.random(N_VENTAS) < 0.01
cantidades[mask_cant_neg] = -cantidades[mask_cant_neg]

# Fechas como strings en formato "%Y-%m-%d"
# Rango: 2024-01-01 a 2025-12-31 (730 días)
base_date = np.datetime64("2024-01-01")
dias_offset = np.random.randint(0, 730, N_VENTAS)
fechas_dt = base_date + dias_offset.astype("timedelta64[D]")
fechas_str = [str(f) for f in fechas_dt]  # "2024-03-15" como string

# Región y sucursal
regiones = np.random.choice(["Norte", "Sur", "Centro", "Este", "Oeste"], N_VENTAS)
sucursales = np.random.choice([f"SUC-{i:03d}" for i in range(1, 51)], N_VENTAS)

# ── Columnas de relleno (padding) — simulan un CSV con columnas innecesarias ──
padding = {}
for i in range(1, 7):
    padding[f"padding_{i:02d}"] = np.random.choice(
        ["x", "y", "z", "info_irrelevante", "dato_legacy"], N_VENTAS
    ).tolist()

# ── Construir DataFrame ──
data = {
    "venta_id":     list(range(1, N_VENTAS + 1)),
    "producto_id":  producto_ids.tolist(),
    "cliente":      clientes,
    "vendedor":     vendedores,
    "precio":       precios.tolist(),
    "cantidad":     cantidades.tolist(),
    "fecha_str":    fechas_str,
    "region":       regiones.tolist(),
    "sucursal":     sucursales.tolist(),
}
data.update(padding)

df_ventas = pl.DataFrame(data)

# ── Introducir ~5% de nulls en columnas clave ──
# Polars no permite asignación directa; usamos when/otherwise
null_rate = 0.05
null_masks = {
    col: np.random.random(N_VENTAS) < null_rate
    for col in ["precio", "cantidad", "cliente", "region"]
}

df_ventas = df_ventas.with_columns([
    pl.when(pl.Series(null_masks["precio"]))
      .then(None)
      .otherwise(pl.col("precio"))
      .alias("precio"),
    pl.when(pl.Series(null_masks["cantidad"]))
      .then(None)
      .otherwise(pl.col("cantidad"))
      .cast(pl.Int64, strict=False)
      .alias("cantidad"),
    pl.when(pl.Series(null_masks["cliente"]))
      .then(None)
      .otherwise(pl.col("cliente"))
      .alias("cliente"),
    pl.when(pl.Series(null_masks["region"]))
      .then(None)
      .otherwise(pl.col("region"))
      .alias("region"),
])

# ── Introducir ~1% de filas duplicadas ──
n_dupes = int(N_VENTAS * 0.01)
idx_dupes = np.random.randint(0, N_VENTAS, n_dupes)
df_dupes = df_ventas[idx_dupes.tolist()]
df_ventas = pl.concat([df_ventas, df_dupes])

# ── Escribir CSV sucio ──
df_ventas.write_csv(path_ventas)

print(f"ventas_sucias.csv: {df_ventas.shape}")
print(f"Columnas: {df_ventas.columns}")
print(f"\nNulls por columna:")
print(df_ventas.null_count())
print(f"\nPrecios negativos: {(df_ventas['precio'].drop_nulls() < 0).sum()}")
print(f"Cantidades negativas: {(df_ventas['cantidad'].drop_nulls() < 0).sum()}")
print(f"\nPrimeras 5 filas:")
df_ventas.head(5)

ventas_sucias.csv: (50500, 15)
Columnas: ['venta_id', 'producto_id', 'cliente', 'vendedor', 'precio', 'cantidad', 'fecha_str', 'region', 'sucursal', 'padding_01', 'padding_02', 'padding_03', 'padding_04', 'padding_05', 'padding_06']

Nulls por columna:
shape: (1, 15)
┌──────────┬────────────┬─────────┬──────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ venta_id ┆ producto_i ┆ cliente ┆ vendedor ┆ … ┆ padding_03 ┆ padding_04 ┆ padding_0 ┆ padding_0 │
│ ---      ┆ d          ┆ ---     ┆ ---      ┆   ┆ ---        ┆ ---        ┆ 5         ┆ 6         │
│ u32      ┆ ---        ┆ u32     ┆ u32      ┆   ┆ u32        ┆ u32        ┆ ---       ┆ ---       │
│          ┆ u32        ┆         ┆          ┆   ┆            ┆            ┆ u32       ┆ u32       │
╞══════════╪════════════╪═════════╪══════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 0        ┆ 0          ┆ 2478    ┆ 0        ┆ … ┆ 0          ┆ 0          ┆ 0         ┆ 0         │
└──────────┴────────────┴

venta_id,producto_id,cliente,vendedor,precio,cantidad,fecha_str,region,sucursal,padding_01,padding_02,padding_03,padding_04,padding_05,padding_06
i64,i64,str,str,f64,i64,str,str,str,str,str,str,str,str,str
1,138,"""luis""","""JUAN PÉREZ""",157.32,19,"""2024-12-04""","""Norte""","""SUC-001""","""y""","""z""","""z""","""dato_legacy""","""y""","""dato_legacy"""
2,181,"""JAIME""","""miguel ángel ruiz""",429.83,6,"""2024-08-04""","""Norte""","""SUC-011""","""y""","""info_irrelevante""","""y""","""x""","""y""","""info_irrelevante"""
3,60,"""luis""","""pedro lópez """,18.24,19,"""2025-01-18""","""Centro""","""SUC-030""","""x""","""x""","""x""","""x""","""y""","""x"""
4,89,""" MARIA """,""" JUAN PÉREZ """,31.51,8,"""2025-12-19""","""Este""","""SUC-017""","""dato_legacy""","""x""","""z""","""dato_legacy""","""z""","""info_irrelevante"""
5,118,""" GABRIELA """,""" Pedro López """,36.74,4,"""2024-10-16""","""Norte""","""SUC-050""","""info_irrelevante""","""z""","""y""","""z""","""info_irrelevante""","""dato_legacy"""


Ahora tenemos dos CSVs en disco con datos realistamente sucios. El pipeline que construiremos
a continuación tomará estos archivos y producirá un resumen limpio y agregado.

**Nota importante:** A partir de aquí, olvidamos que los DataFrames `df_ventas` y `df_productos`
existen. Trabajamos **solo** con los archivos CSV, como si los recibiéramos de un sistema externo.
Limpiamos la memoria para ser honestos:

In [6]:
# Eliminar DataFrames de memoria — trabajamos desde los CSVs
del df_ventas, df_productos, data, padding
print("DataFrames eliminados. Trabajamos desde los CSVs.")

DataFrames eliminados. Trabajamos desde los CSVs.


---

## §2: Scan — registrar, no leer

El primer paso del pipeline es **registrar** los archivos CSV con `pl.scan_csv()`.
Esta función **no lee un solo byte** del archivo. Solo crea un nodo `SCAN` en el
plan de ejecución, que dice: "cuando llegue el momento, lee este archivo".

Esto es fundamentalmente diferente de `pd.read_csv()` de pandas, que:
1. Abre el archivo
2. Lee **todo** el contenido en memoria
3. Infiere tipos de **todas** las columnas
4. Construye un DataFrame completo

Con `scan_csv`, Polars solo inspecciona las primeras filas para inferir el esquema
(nombres y tipos de columnas), pero no materializa ningún dato.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica la diferencia entre
> `scan_csv` y `pd.read_csv`, y por qué el modo lazy es más eficiente.

In [7]:
# ── Paso 1: Registrar los archivos — NO leer ──
lf_ventas = pl.scan_csv(path_ventas)
lf_productos = pl.scan_csv(path_productos)

# Verificar: esto es un LazyFrame, NO un DataFrame
print(f"Tipo lf_ventas:    {type(lf_ventas)}")
print(f"Tipo lf_productos: {type(lf_productos)}")

# Podemos ver el esquema sin leer datos
print(f"\nEsquema ventas ({len(lf_ventas.collect_schema())} columnas):")
for nombre, tipo in lf_ventas.collect_schema().items():
    print(f"  {nombre:20s} → {tipo}")

Tipo lf_ventas:    <class 'polars.lazyframe.frame.LazyFrame'>
Tipo lf_productos: <class 'polars.lazyframe.frame.LazyFrame'>

Esquema ventas (15 columnas):
  venta_id             → Int64
  producto_id          → Int64
  cliente              → String
  vendedor             → String
  precio               → Float64
  cantidad             → Int64
  fecha_str            → String
  region               → String
  sucursal             → String
  padding_01           → String
  padding_02           → String
  padding_03           → String
  padding_04           → String
  padding_05           → String
  padding_06           → String


In [8]:
# ── Plan de ejecución: solo un nodo SCAN ──
print("Plan de ejecución (solo scan):")
print(lf_ventas.explain())

Plan de ejecución (solo scan):
Csv SCAN [/tmp/polars_pipeline_yqkw4qyg/ventas_sucias.csv]
PROJECT */15 COLUMNS


### ¿Qué nos dice `.explain()`?

El plan muestra un único nodo **CSV SCAN**. Esto confirma que Polars no ha leído
nada todavía — solo tiene registrada la intención de leer el archivo.

Observa que el plan ya menciona las columnas disponibles. Polars leyó solo las
primeras filas del CSV para inferir el esquema, pero no cargó los datos.

Conforme agreguemos transformaciones, el plan crecerá con nuevos nodos, y el
optimizador los reorganizará para ejecutar todo de la forma más eficiente posible.

---

## §3: Validación de esquema y casting

Los CSVs no tienen tipos fuertes — todo llega como `String` o como un tipo inferido
que puede no ser el correcto. El primer paso de cualquier pipeline serio es **forzar
los tipos correctos** mediante casting explícito.

En Polars, los casts son estrictos por defecto: si un valor no es convertible, el
pipeline falla con un error claro. Esto es mejor que las conversiones silenciosas
de pandas, donde un `NaN` puede aparecer sin aviso.

También aprovechamos este paso para **seleccionar** solo las columnas que nos interesan,
descartando las 6 columnas de relleno (`padding_*`). El optimizador usará esto para
aplicar **projection pushdown**: ni siquiera leerá esas columnas del CSV.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` muestra el casting de tipos
> como el paso 2 del pipeline.

In [9]:
# ── Paso 2: Seleccionar columnas útiles y castear tipos ──
lf_ventas = (
    lf_ventas
    # Seleccionar solo las columnas que necesitamos (descartamos padding_*)
    .select(
        "venta_id",
        "producto_id",
        "cliente",
        "vendedor",
        "precio",
        "cantidad",
        "fecha_str",
        "region",
        "sucursal",
    )
    # Castear a tipos correctos
    .with_columns(
        pl.col("precio").cast(pl.Float64),      # asegurar flotante de 64 bits
        pl.col("cantidad").cast(pl.Int32),       # entero de 32 bits
        pl.col("producto_id").cast(pl.Int64),    # entero de 64 bits
        pl.col("venta_id").cast(pl.Int64),       # entero de 64 bits
    )
)

# ── Inspeccionar el plan ──
print("Plan después de select + cast:")
print(lf_ventas.explain())

Plan después de select + cast:
 WITH_COLUMNS:
 [col("precio").strict_cast(Float64), col("cantidad").strict_cast(Int32), col("producto_id").strict_cast(Int64), col("venta_id").strict_cast(Int64)] 
  Csv SCAN [/tmp/polars_pipeline_yqkw4qyg/ventas_sucias.csv]
  PROJECT 9/15 COLUMNS


### ¿Qué cambió en el plan?

Ahora el plan tiene dos cambios importantes:

1. **PROJECT**: El nodo de scan ahora indica que solo se leerán 9 de las 15 columnas
   originales. Las 6 columnas `padding_*` nunca serán leídas del disco.
   Esto es **projection pushdown** — el optimizador "empuja" la selección de columnas
   hasta el nodo de lectura.

2. **WITH_COLUMNS / CAST**: Aparece un nodo que representa los casts de tipos.

El punto clave: aunque escribimos `.select()` y `.with_columns()` como pasos separados,
el optimizador los puede fusionar y aplicar directamente en la lectura.

---

## §4: Filtrar filas inválidas

Nuestros datos tienen varios tipos de filas inválidas:

- **Precios negativos** — errores de captura donde el signo se invirtió
- **Cantidades negativas o cero** — no tiene sentido en ventas
- **Nulls en campos críticos** — precio y cantidad son esenciales para el análisis
- **Filas duplicadas** — registros repetidos que inflarían las métricas

En el plan de ejecución, el filtro aparecerá como un nodo `FILTER` (o `SELECTION`).
El optimizador intentará aplicar **predicate pushdown**: fusionar el filtro con el
nodo de lectura para descartar filas lo antes posible.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica el filtrado y el
> predicate pushdown en el contexto del pipeline.

In [10]:
# ── Paso 3: Filtrar filas inválidas ──
lf_ventas = (
    lf_ventas
    # Eliminar filas con nulls en campos críticos
    .filter(
        pl.col("precio").is_not_null()
        & pl.col("cantidad").is_not_null()
        & pl.col("cliente").is_not_null()
    )
    # Eliminar precios y cantidades inválidas
    .filter(
        (pl.col("precio") > 0)       # precios negativos = error de captura
        & (pl.col("cantidad") > 0)   # cantidades <= 0 no tienen sentido
    )
    # Eliminar duplicados exactos
    .unique()
)

# ── Inspeccionar el plan ──
print("Plan después de filtros:")
print(lf_ventas.explain())

Plan después de filtros:
UNIQUE[maintain_order: false, keep_strategy: Any] BY None
  FILTER [([(col("cantidad").is_not_null()) & ([(col("cantidad")) > (0)])]) & ([(col("precio").is_not_null()) & ([(col("precio")) > (0.0)])])]
  FROM
     WITH_COLUMNS:
     [col("precio").strict_cast(Float64), col("cantidad").strict_cast(Int32), col("producto_id").strict_cast(Int64), col("venta_id").strict_cast(Int64)] 
      Csv SCAN [/tmp/polars_pipeline_yqkw4qyg/ventas_sucias.csv]
      PROJECT 9/15 COLUMNS
      SELECTION: col("cliente").is_not_null()


### ¿Qué nos dice `.explain()`?

Ahora vemos nodos de **FILTER** (o **SELECTION**) en el plan. Lo interesante es
observar cómo el optimizador maneja los dos `.filter()` que escribimos por separado:

- **Predicate pushdown**: El optimizador puede fusionar ambos filtros en uno solo
  y aplicarlos lo más cerca posible de la lectura.
- **UNIQUE/DISTINCT**: La deduplicación aparece como un nodo separado porque
  requiere ver todas las filas antes de decidir cuáles son duplicadas.

Nota que escribimos dos `.filter()` encadenados por legibilidad. Polars los fusiona
automáticamente — no hay penalización por separar filtros en pasos legibles.

---

## §5: Limpiar strings

Los strings sucios son uno de los problemas más comunes en datos reales.
Nuestras columnas `cliente` y `vendedor` tienen:

- Espacios al inicio y al final: `" alice "` → `"alice"`
- Mayúsculas inconsistentes: `"ALICE"`, `"alice"`, `"Alice"`

Polars ofrece operaciones `.str.*` que se ejecutan **completamente en Rust**,
sin pasar por Python. Esto es significativamente más rápido que pandas, donde
`df["col"].str.lower()` itera sobre objetos Python internamente.

Las operaciones que usamos:
- `.str.strip_chars()` — elimina espacios al inicio y al final
- `.str.to_lowercase()` — normaliza a minúsculas
- `.str.to_titlecase()` — capitaliza la primera letra de cada palabra

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` muestra exactamente estas
> operaciones de limpieza de strings.

In [11]:
# ── Paso 4: Limpiar strings ──
lf_ventas = lf_ventas.with_columns(
    # Cliente: quitar espacios, normalizar a minúsculas, luego capitalizar
    pl.col("cliente")
        .str.strip_chars()           # "  alice  " → "alice"
        .str.to_lowercase()          # "ALICE" → "alice"
        .str.to_titlecase()          # "alice" → "Alice"
        .alias("cliente"),

    # Vendedor: quitar espacios, normalizar a title case
    pl.col("vendedor")
        .str.strip_chars()           # " juan pérez " → "juan pérez"
        .str.to_titlecase()          # "juan pérez" → "Juan Pérez"
        .alias("vendedor"),

    # Región: normalizar a title case
    pl.col("region")
        .str.strip_chars()
        .str.to_titlecase()
        .alias("region"),
)

# ── Inspeccionar el plan ──
print("Plan después de limpieza de strings:")
print(lf_ventas.explain())

Plan después de limpieza de strings:
 WITH_COLUMNS:
 [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("vendedor").str.strip_chars([null]).str.titlecase().alias("vendedor"), col("region").str.strip_chars([null]).str.titlecase().alias("region")] 
  UNIQUE[maintain_order: false, keep_strategy: Any] BY None
    FILTER [([(col("precio").is_not_null()) & ([(col("precio")) > (0.0)])]) & ([(col("cantidad").is_not_null()) & ([(col("cantidad")) > (0)])])]
    FROM
       WITH_COLUMNS:
       [col("precio").strict_cast(Float64), col("cantidad").strict_cast(Int32), col("producto_id").strict_cast(Int64), col("venta_id").strict_cast(Int64)] 
        Csv SCAN [/tmp/polars_pipeline_yqkw4qyg/ventas_sucias.csv]
        PROJECT 9/15 COLUMNS
        SELECTION: col("cliente").is_not_null()


### ¿Qué nos dice `.explain()`?

El plan ahora incluye un nodo **WITH_COLUMNS** que contiene las operaciones de
limpieza de strings. Puntos clave:

1. **Todas las operaciones `.str.*` son expresiones nativas** — se compilan a Rust
   y se ejecutan sin overhead de Python. No hay serialización ni deserialización.

2. **Las tres transformaciones (strip, lowercase, titlecase) se encadenan** en una
   sola expresión por columna. Polars las ejecuta secuencialmente sobre cada valor,
   sin crear strings intermedios innecesarios.

3. **Las columnas `cliente`, `vendedor` y `region` se procesan en paralelo** —
   Polars puede usar múltiples cores para procesar columnas independientes.

---

## §6: Parsear fechas y extraer componentes

Nuestra columna `fecha_str` contiene fechas como strings (`"2024-03-15"`).
Necesitamos:

1. **Convertir** el string a tipo `Date` nativo de Polars
2. **Extraer** componentes temporales: año, mes, día de la semana

El namespace `.dt.*` de Polars da acceso a todas las operaciones temporales:
extracción de componentes, aritmética de fechas, truncamiento, etc.

Todo esto se ejecuta en Rust sobre la representación interna de Arrow (días desde
epoch como enteros de 32 bits), lo que es enormemente más eficiente que las
operaciones de `datetime` de Python.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` muestra el parseo de fechas
> y la extracción de componentes con `.dt.*`.

In [12]:
# ── Paso 5: Parsear fechas y extraer componentes ──
lf_ventas = (
    lf_ventas
    # Convertir string → Date
    .with_columns(
        pl.col("fecha_str")
            .str.to_date("%Y-%m-%d")    # "2024-03-15" → Date(2024, 3, 15)
            .alias("fecha"),
    )
    # Extraer componentes temporales
    .with_columns(
        pl.col("fecha").dt.year().alias("anio"),           # 2024
        pl.col("fecha").dt.month().alias("mes"),           # 3
        pl.col("fecha").dt.weekday().alias("dia_semana"),  # 1=lunes, 7=domingo
    )
    # Ya no necesitamos la columna string original
    .drop("fecha_str")
)

# ── Inspeccionar el plan ──
print("Plan después de parsear fechas:")
print(lf_ventas.explain())

Plan después de parsear fechas:
simple π 12/13 ["venta_id", "producto_id", ... 10 other columns]
   WITH_COLUMNS:
   [col("fecha").dt.year().alias("anio"), col("fecha").dt.month().alias("mes"), col("fecha").dt.weekday().alias("dia_semana")] 
     WITH_COLUMNS:
     [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("vendedor").str.strip_chars([null]).str.titlecase().alias("vendedor"), col("region").str.strip_chars([null]).str.titlecase().alias("region"), col("fecha_str").str.strptime(["raise"]).alias("fecha")] 
      UNIQUE[maintain_order: false, keep_strategy: Any] BY None
        FILTER [([(col("cantidad").is_not_null()) & ([(col("cantidad")) > (0)])]) & ([(col("precio").is_not_null()) & ([(col("precio")) > (0.0)])])]
        FROM
           WITH_COLUMNS:
           [col("precio").strict_cast(Float64), col("cantidad").strict_cast(Int32), col("producto_id").strict_cast(Int64), col("venta_id").strict_cast(Int64)] 
            Csv SCAN [/tmp/po

### ¿Qué nos dice `.explain()`?

El plan ahora incluye:

1. **WITH_COLUMNS** para el parseo `str → Date` — una sola operación que convierte
   el string a la representación interna de Arrow.

2. **WITH_COLUMNS** para la extracción de componentes — `.dt.year()`, `.dt.month()`,
   `.dt.weekday()` son operaciones aritméticas simples sobre enteros internos.

3. **SIMPLE_PROJECTION** (o similar) para el `.drop("fecha_str")` — el optimizador
   puede incluso eliminar la lectura de `fecha_str` del CSV si ya no se necesita
   después de crear `fecha`. Pero como `fecha` se **deriva** de `fecha_str`, la
   columna debe leerse; lo que no se mantendrá en memoria después de la conversión.

Observa cómo usamos dos `.with_columns()` encadenados: el primero crea `fecha`,
el segundo usa `fecha` para extraer componentes. Este orden es necesario porque
las expresiones dentro de un mismo `.with_columns()` se ejecutan en paralelo y
no pueden referirse a columnas creadas en el mismo paso.

---

## §7: Join con catálogo de productos

Hasta ahora hemos trabajado solo con `ventas_sucias.csv`. Ahora enriquecemos los
datos haciendo un **left join** con el catálogo de productos.

El join nos dará acceso a:
- `categoria` — para agrupar por categoría de producto
- `subcategoria` — detalle adicional
- `proveedor` — información del proveedor

Lo interesante desde el punto de vista del optimizador es el **projection pushdown
a través del join**: si después del join solo usamos `categoria` y `subcategoria`
del catálogo, Polars no leerá la columna `proveedor` del CSV de productos.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica cómo el optimizador
> aplica projection pushdown en ambos lados del join.

In [13]:
# ── Paso 6: Join con catálogo de productos ──
lf_enriquecido = lf_ventas.join(
    lf_productos,
    on="producto_id",     # columna de unión
    how="left",           # left join: mantener todas las ventas
)

# ── Inspeccionar el plan — observar projection pushdown en AMBOS lados ──
print("Plan después del join:")
print(lf_enriquecido.explain())

Plan después del join:
LEFT JOIN:
LEFT PLAN ON: [col("producto_id")]
  simple π 12/13 ["venta_id", "producto_id", ... 10 other columns]
     WITH_COLUMNS:
     [col("fecha").dt.year().alias("anio"), col("fecha").dt.month().alias("mes"), col("fecha").dt.weekday().alias("dia_semana")] 
       WITH_COLUMNS:
       [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("vendedor").str.strip_chars([null]).str.titlecase().alias("vendedor"), col("region").str.strip_chars([null]).str.titlecase().alias("region"), col("fecha_str").str.strptime(["raise"]).alias("fecha")] 
        UNIQUE[maintain_order: false, keep_strategy: Any] BY None
          FILTER [([(col("cantidad").is_not_null()) & ([(col("cantidad")) > (0)])]) & ([(col("precio").is_not_null()) & ([(col("precio")) > (0.0)])])]
          FROM
             WITH_COLUMNS:
             [col("precio").strict_cast(Float64), col("cantidad").strict_cast(Int32), col("producto_id").strict_cast(Int64), col("vent

### ¿Qué nos dice `.explain()`?

El plan ahora muestra un nodo **JOIN** con dos ramas de entrada:

1. **Rama izquierda** (ventas): incluye todos los nodos que hemos construido —
   scan, cast, filter, string cleanup, date parsing.

2. **Rama derecha** (productos): un scan separado del CSV de productos.

El punto clave es el **projection pushdown a través del join**. Observa qué
columnas de `productos` aparecen en el plan:

- Si el plan final solo usa `categoria` (para la agregación), el optimizador
  debería indicar que solo lee `producto_id` y `categoria` del catálogo.
- Las columnas `subcategoria` y `proveedor` podrían no leerse del disco.

Esto es enormemente eficiente en joins con tablas anchas: si el catálogo tuviera
100 columnas pero solo necesitas 2, Polars solo lee esas 2.

---

## §8: Agregación

El paso final de transformación es la **agregación**: resumir las transacciones
individuales en métricas por categoría y mes.

Usamos `group_by()` + `.agg()` con múltiples expresiones de agregación:

| Expresión | Descripción |
|-----------|-------------|
| `pl.col("precio").sum()` | Revenue total |
| `pl.col("precio").mean()` | Ticket promedio |
| `pl.col("producto_id").n_unique()` | Productos distintos vendidos |
| `pl.len()` | Número de transacciones |
| `pl.col("cantidad").sum()` | Unidades vendidas |
| `pl.col("cliente").n_unique()` | Clientes únicos |

Todas las expresiones dentro de `.agg()` se ejecutan en paralelo sobre cada grupo.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` muestra la misma estructura
> de group_by + agg.

In [14]:
# ── Paso 7: Agregación por categoría y mes ──
lf_resultado = (
    lf_enriquecido
    .group_by("categoria", "mes")
    .agg(
        # Revenue total por grupo
        pl.col("precio").sum().alias("revenue"),
        # Ticket promedio
        pl.col("precio").mean().alias("ticket_promedio"),
        # Productos distintos vendidos
        pl.col("producto_id").n_unique().alias("productos_distintos"),
        # Total de transacciones
        pl.len().alias("n_transacciones"),
        # Unidades vendidas
        pl.col("cantidad").sum().alias("unidades_vendidas"),
        # Clientes únicos
        pl.col("cliente").n_unique().alias("clientes_unicos"),
    )
    .sort("categoria", "mes")  # ordenar para legibilidad
)

# ── Inspeccionar el plan ──
print("Plan después de la agregación:")
print(lf_resultado.explain())

Plan después de la agregación:
SORT BY [col("categoria"), col("mes")]
  AGGREGATE
    [col("precio").sum().alias("revenue"), col("precio").mean().alias("ticket_promedio"), col("producto_id").n_unique().alias("productos_distintos"), len().alias("n_transacciones"), col("cantidad").sum().alias("unidades_vendidas"), col("cliente").n_unique().alias("clientes_unicos")] BY [col("categoria"), col("mes")]
    FROM
    simple π 6/6 ["producto_id", "precio", ... 4 other columns]
      LEFT JOIN:
      LEFT PLAN ON: [col("producto_id")]
        simple π 5/11 ["producto_id", "cliente", ... 3 other columns]
           WITH_COLUMNS:
           [col("fecha").dt.month().alias("mes")] 
             WITH_COLUMNS:
             [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("fecha_str").str.strptime(["raise"]).alias("fecha")] 
              UNIQUE[maintain_order: false, keep_strategy: Any] BY None
                FILTER [([(col("cantidad").is_not_null()) & ([(

### ¿Qué nos dice `.explain()`?

El plan ahora incluye un nodo **AGGREGATE** (o `GROUP_BY`) que:

1. Agrupa por `categoria` y `mes`
2. Calcula 6 expresiones de agregación en paralelo
3. Seguido de un **SORT** para ordenar el resultado

La agregación es uno de los puntos donde Polars brilla más:
- Cada expresión de agregación es independiente y se puede paralelizar
- El formato columnar de Arrow es ideal para operaciones como `sum()` y `mean()`
  porque los datos de una columna están contiguos en memoria (cache-friendly)
- No hay conversión Python ↔ C como en pandas

Nota que el **projection pushdown** ahora tiene efecto completo: como la agregación
solo usa `precio`, `producto_id`, `cantidad`, `cliente`, `categoria` y `mes`, el
optimizador sabe que columnas como `vendedor`, `sucursal`, `region`, `subcategoria`
y `proveedor` **nunca se necesitan** y puede evitar leerlas.

---

## §9: Plan final — optimizado vs no optimizado

Este es el momento más revelador del pipeline. Vamos a comparar:

1. **Plan no optimizado** (`optimized=False`) — exactamente lo que escribimos,
   paso por paso, sin reordenar ni fusionar nada.

2. **Plan optimizado** (`optimized=True`, el default) — lo que Polars realmente
   va a ejecutar, después de aplicar todas sus reglas de optimización.

Las diferencias clave que buscar:

| Optimización | Qué hace | Efecto |
|-------------|----------|--------|
| Projection pushdown | Solo lee las columnas necesarias | Menos I/O, menos memoria |
| Predicate pushdown | Aplica filtros lo antes posible | Menos filas procesadas |
| Type coercion | Fusiona casts con la lectura | Menos copias |
| Common subexpression elimination | Reutiliza resultados intermedios | Menos cálculos |

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` dedica el paso 8 a comparar
> los planes optimizado y no optimizado.

In [15]:
# ── Plan NO optimizado: lo que escribimos ──
print("=" * 70)
print("PLAN NO OPTIMIZADO (lo que escribimos)")
print("=" * 70)
print(lf_resultado.explain(optimized=False))

PLAN NO OPTIMIZADO (lo que escribimos)
SORT BY [col("categoria"), col("mes")]
  AGGREGATE
    [col("precio").sum().alias("revenue"), col("precio").mean().alias("ticket_promedio"), col("producto_id").n_unique().alias("productos_distintos"), len().alias("n_transacciones"), col("cantidad").sum().alias("unidades_vendidas"), col("cliente").n_unique().alias("clientes_unicos")] BY [col("categoria"), col("mes")]
    FROM
    LEFT JOIN:
    LEFT PLAN ON: [col("producto_id")]
      simple π 12/13 ["venta_id", "producto_id", ... 10 other columns]
         WITH_COLUMNS:
         [col("fecha").dt.year().alias("anio"), col("fecha").dt.month().alias("mes"), col("fecha").dt.weekday().alias("dia_semana")] 
           WITH_COLUMNS:
           [col("fecha_str").str.strptime(["raise"]).alias("fecha")] 
             WITH_COLUMNS:
             [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("vendedor").str.strip_chars([null]).str.titlecase().alias("vendedor"), c

In [16]:
# ── Plan OPTIMIZADO: lo que Polars ejecutará ──
print("=" * 70)
print("PLAN OPTIMIZADO (lo que Polars ejecutará)")
print("=" * 70)
print(lf_resultado.explain(optimized=True))

PLAN OPTIMIZADO (lo que Polars ejecutará)
SORT BY [col("categoria"), col("mes")]
  AGGREGATE
    [col("precio").sum().alias("revenue"), col("precio").mean().alias("ticket_promedio"), col("producto_id").n_unique().alias("productos_distintos"), len().alias("n_transacciones"), col("cantidad").sum().alias("unidades_vendidas"), col("cliente").n_unique().alias("clientes_unicos")] BY [col("categoria"), col("mes")]
    FROM
    simple π 6/6 ["producto_id", "precio", ... 4 other columns]
      LEFT JOIN:
      LEFT PLAN ON: [col("producto_id")]
        simple π 5/11 ["producto_id", "cliente", ... 3 other columns]
           WITH_COLUMNS:
           [col("fecha").dt.month().alias("mes")] 
             WITH_COLUMNS:
             [col("cliente").str.strip_chars([null]).str.lowercase().str.titlecase().alias("cliente"), col("fecha_str").str.strptime(["raise"]).alias("fecha")] 
              UNIQUE[maintain_order: false, keep_strategy: Any] BY None
                FILTER [([(col("cantidad").is_not_nu

In [17]:
# ── Análisis: ¿cuántas columnas se leen realmente? ──
plan_opt = lf_resultado.explain(optimized=True)

# Contar las columnas originales del CSV de ventas (15 columnas)
columnas_originales = [
    "venta_id", "producto_id", "cliente", "vendedor", "precio",
    "cantidad", "fecha_str", "region", "sucursal",
    "padding_01", "padding_02", "padding_03",
    "padding_04", "padding_05", "padding_06",
]

# Columnas que el pipeline final realmente necesita
columnas_usadas = [
    "producto_id", "cliente", "precio", "cantidad", "fecha_str"
]

print(f"Columnas en el CSV original:     {len(columnas_originales)}")
print(f"Columnas que realmente se usan:  {len(columnas_usadas)}")
print(f"Columnas que NO se leen:         {len(columnas_originales) - len(columnas_usadas)}")
print(f"\nAhorro en lectura: {(1 - len(columnas_usadas)/len(columnas_originales))*100:.0f}% menos columnas")
print(f"\nColumnas necesarias: {columnas_usadas}")
print(f"\nColumnas descartadas por projection pushdown:")
for c in columnas_originales:
    if c not in columnas_usadas:
        print(f"  ✗ {c}")

Columnas en el CSV original:     15
Columnas que realmente se usan:  5
Columnas que NO se leen:         10

Ahorro en lectura: 67% menos columnas

Columnas necesarias: ['producto_id', 'cliente', 'precio', 'cantidad', 'fecha_str']

Columnas descartadas por projection pushdown:
  ✗ venta_id
  ✗ vendedor
  ✗ region
  ✗ sucursal
  ✗ padding_01
  ✗ padding_02
  ✗ padding_03
  ✗ padding_04
  ✗ padding_05
  ✗ padding_06


### Análisis detallado de las diferencias

Comparando ambos planes, las diferencias principales son:

**1. Projection pushdown (columnas)**
- Plan no optimizado: lee las 15 columnas del CSV, luego descarta las que no necesita
- Plan optimizado: solo lee ~5 columnas del CSV (las que realmente llegan al resultado final)
- Impacto: **~67% menos datos leídos del disco**

**2. Predicate pushdown (filas)**
- Plan no optimizado: lee todas las filas, luego aplica filtros
- Plan optimizado: los filtros (`precio > 0`, `is_not_null`) se aplican lo más
  cerca posible del scan
- Impacto: menos filas pasan por las transformaciones posteriores

**3. Fusion de nodos**
- Los dos `.filter()` separados se fusionan en uno solo
- Los `.with_columns()` de cast y limpieza pueden fusionarse

**4. Projection pushdown en el join**
- Del catálogo de productos (4 columnas), solo se leen `producto_id` y `categoria`
- `subcategoria` y `proveedor` nunca se leen del disco

Este es el poder del modo lazy: nosotros escribimos código legible y modular,
y el optimizador lo reescribe para ser eficiente.

---

## §10: `.collect()` — ejecutar todo

Hasta este punto, **no se ha ejecutado nada**. Todo ha sido planificación.
`.collect()` es el momento donde Polars:

1. Toma el plan optimizado
2. Lee los CSVs (solo las columnas necesarias)
3. Aplica filtros, transformaciones, join, agregación
4. Produce un `DataFrame` materializado en memoria

Todo esto ocurre en una sola pasada, paralelizada, sin DataFrames intermedios.

Compara con el equivalente en pandas, donde cada línea crearía un DataFrame
nuevo en memoria:

```python
# pandas: cada línea materializa un DataFrame completo
df = pd.read_csv("ventas.csv")                        # lee TODO, 15 cols
df = df[["venta_id", "producto_id", ...]]             # copia
df = df.dropna(subset=["precio", "cantidad"])          # copia
df = df[df["precio"] > 0]                              # copia
df["cliente"] = df["cliente"].str.strip().str.lower()   # modifica (a veces)
df["fecha"] = pd.to_datetime(df["fecha_str"])           # copia
df = df.merge(productos, on="producto_id")              # copia
resultado = df.groupby(["cat", "mes"]).agg(...)         # copia
# Total: ~6-8 copias completas del DataFrame en memoria
```

Con Polars lazy, hay **cero** DataFrames intermedios.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica que todo el pipeline
> se ejecuta en una sola pasada con `.collect()`.

In [18]:
# ── Ejecutar todo el pipeline ──
t0 = time.perf_counter()
df_resultado = lf_resultado.collect()
t1 = time.perf_counter()

print(f"Tiempo de ejecución: {(t1 - t0)*1000:.1f} ms")
print(f"Forma del resultado: {df_resultado.shape}")
print(f"Columnas: {df_resultado.columns}")
print(f"\nEsquema:")
for nombre, tipo in df_resultado.schema.items():
    print(f"  {nombre:25s} → {tipo}")

Tiempo de ejecución: 34.5 ms
Forma del resultado: (60, 8)
Columnas: ['categoria', 'mes', 'revenue', 'ticket_promedio', 'productos_distintos', 'n_transacciones', 'unidades_vendidas', 'clientes_unicos']

Esquema:
  categoria                 → String
  mes                       → Int8
  revenue                   → Float64
  ticket_promedio           → Float64
  productos_distintos       → UInt32
  n_transacciones           → UInt32
  unidades_vendidas         → Int32
  clientes_unicos           → UInt32


In [19]:
# ── Mostrar el resultado completo ──
print("Resultado del pipeline (primeras 20 filas):")
df_resultado.head(20)

Resultado del pipeline (primeras 20 filas):


categoria,mes,revenue,ticket_promedio,productos_distintos,n_transacciones,unidades_vendidas,clientes_unicos
str,i8,f64,f64,u32,u32,i32,u32
"""Alimentos""",1,99534.41,154.316915,36,645,6495,15
"""Alimentos""",2,87595.24,148.466508,36,590,6186,15
"""Alimentos""",3,98019.47,160.162533,36,612,6300,15
"""Alimentos""",4,92285.96,143.30118,36,644,6960,15
"""Alimentos""",5,101274.07,159.235959,36,636,6622,15
…,…,…,…,…,…,…,…
"""Deportes""",4,119259.02,148.332114,49,804,8518,15
"""Deportes""",5,129217.21,149.730255,49,863,9328,15
"""Deportes""",6,133980.42,157.624024,49,850,9022,15


In [20]:
# ── Estadísticas descriptivas del resultado ──
print("Estadísticas descriptivas:")
df_resultado.describe()

Estadísticas descriptivas:


statistic,categoria,mes,revenue,ticket_promedio,productos_distintos,n_transacciones,unidades_vendidas,clientes_unicos
str,str,f64,f64,f64,f64,f64,f64,f64
"""count""","""60""",60.0,60.0,60.0,60.0,60.0,60.0,60.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,6.5,104305.903,150.250886,40.0,694.433333,7266.216667,15.0
"""std""",null,3.481184,14283.813528,5.588541,5.336157,93.603811,977.904454,0.0
"""min""","""Alimentos""",1.0,81633.95,139.185164,35.0,553.0,5841.0,15.0
"""25%""",null,4.0,94011.68,145.987361,36.0,627.0,6501.0,15.0
"""50%""",null,7.0,100917.66,149.730255,37.0,666.0,6960.0,15.0
"""75%""",null,9.0,113042.0,153.402488,43.0,759.0,7927.0,15.0
"""max""","""Ropa""",12.0,136385.34,160.660134,49.0,898.0,9328.0,15.0


### Observaciones sobre la ejecución

Todo el pipeline — lectura de 2 CSVs, casting, filtrado, limpieza de strings,
parseo de fechas, join, agregación y ordenamiento — se ejecutó en una sola
llamada a `.collect()`.

**¿Qué hubiera pasado con pandas?**

Con 50K filas, la diferencia es pequeña. Pero con datos reales (millones de filas,
decenas de columnas), las diferencias se acumulan:

- **Memoria**: pandas crearía ~6-8 copias intermedias del DataFrame.
  Con 10M filas y 50 columnas, eso puede ser GBs de memoria desperdiciada.
- **I/O**: pandas leería las 15 columnas del CSV completas.
  Polars solo leyó ~5 columnas gracias al projection pushdown.
- **Paralelismo**: pandas es single-threaded por defecto.
  Polars paraleliza automáticamente las operaciones por columna.
- **Tipos**: pandas usa objetos Python para strings (lento).
  Polars usa Arrow strings (contiguo en memoria, Rust nativo).

---

## §11: Escribir resultado

El último paso es persistir el resultado en **Parquet**, el formato ideal para
datos tabulares procesados.

### ¿Por qué Parquet y no CSV?

| Característica | CSV | Parquet |
|---------------|-----|--------|
| **Formato** | Texto plano | Binario columnar |
| **Tipos** | Ninguno (todo es string) | Embebidos en el archivo |
| **Compresión** | No nativa | Snappy/Zstd/LZ4 integrado |
| **Lectura parcial** | Leer todo siempre | Leer solo columnas/grupos necesarios |
| **Interoperabilidad** | Universal | Polars, pandas, Spark, DuckDB, Arrow |
| **Tamaño** | Grande | 2-10x más pequeño |

Parquet preserva los tipos, comprime automáticamente, y permite lectura selectiva
de columnas — todo lo que un CSV no puede hacer.

> **Referencia:** Sección 17.4 en `04_pipeline_e2e.md` explica por qué Parquet es
> el formato ideal de salida para pipelines de datos.

In [21]:
# ── Escribir resultado en Parquet ──
df_resultado.write_parquet(path_resultado)

# Comparar tamaños
size_ventas_csv = os.path.getsize(path_ventas)
size_productos_csv = os.path.getsize(path_productos)
size_parquet = os.path.getsize(path_resultado)

print(f"Tamaños de archivo:")
print(f"  ventas_sucias.csv:   {size_ventas_csv / 1024:.1f} KB (entrada, {N_VENTAS}+ filas, 15 columnas)")
print(f"  productos.csv:       {size_productos_csv / 1024:.1f} KB (entrada, {N_PRODUCTOS} filas, 4 columnas)")
print(f"  resumen.parquet:     {size_parquet / 1024:.1f} KB (salida, {df_resultado.shape[0]} filas, {df_resultado.shape[1]} columnas)")
print(f"\nReducción: de {(size_ventas_csv + size_productos_csv)/1024:.0f} KB de entrada a {size_parquet/1024:.1f} KB de salida")

Tamaños de archivo:
  ventas_sucias.csv:   5310.0 KB (entrada, 50000+ filas, 15 columnas)
  productos.csv:       5.4 KB (entrada, 200 filas, 4 columnas)
  resumen.parquet:     4.4 KB (salida, 60 filas, 8 columnas)

Reducción: de 5315 KB de entrada a 4.4 KB de salida


In [22]:
# ── Verificar: releer el Parquet y confirmar que los datos están intactos ──
df_verificacion = pl.read_parquet(path_resultado)

# Los tipos se preservaron — no hay inferencia necesaria
print("Esquema releído desde Parquet:")
for nombre, tipo in df_verificacion.schema.items():
    print(f"  {nombre:25s} → {tipo}")

# Verificar igualdad
print(f"\n¿Datos idénticos? {df_resultado.equals(df_verificacion)}")

Esquema releído desde Parquet:
  categoria                 → String
  mes                       → Int8
  revenue                   → Float64
  ticket_promedio           → Float64
  productos_distintos       → UInt32
  n_transacciones           → UInt32
  unidades_vendidas         → Int32
  clientes_unicos           → UInt32

¿Datos idénticos? True


In [23]:
# ── Limpiar archivos temporales ──
import shutil

shutil.rmtree(tmpdir)
print(f"Directorio temporal eliminado: {tmpdir}")
print("\nPipeline completo.")

Directorio temporal eliminado: /tmp/polars_pipeline_yqkw4qyg

Pipeline completo.


---

## Resumen

En este notebook construimos un pipeline E2E completo en Polars que:

1. **Generó** datos sintéticos sucios (~50K filas, 15 columnas) con problemas
   realistas: nulls, strings sucios, precios negativos, columnas innecesarias.

2. **Registró** los archivos con `scan_csv()` sin leer un solo byte.

3. **Validó y casteó** tipos, seleccionando solo las columnas útiles.

4. **Filtró** filas inválidas (nulls, negativos, duplicados).

5. **Limpió** strings con operaciones nativas de Rust.

6. **Parseó** fechas y extrajo componentes temporales.

7. **Enriqueció** datos con un join al catálogo de productos.

8. **Agregó** métricas por categoría y mes.

9. **Comparó** los planes optimizado y no optimizado, verificando:
   - Projection pushdown: solo ~5 de 15 columnas leídas
   - Predicate pushdown: filtros fusionados con la lectura
   - Projection pushdown en el join: solo columnas necesarias del catálogo

10. **Ejecutó** todo con `.collect()` en una sola pasada.

11. **Persistió** el resultado en Parquet con tipos preservados y compresión.

### La lección clave

Polars nos permite escribir código **legible y modular** (un paso por transformación)
sin sacrificar rendimiento. El optimizador reescribe nuestro plan para:
- Leer solo los datos necesarios
- Filtrar lo antes posible
- Paralelizar operaciones independientes
- Evitar copias intermedias

Nosotros nos enfocamos en la **semántica** (qué queremos hacer), y Polars se
encarga de la **ejecución eficiente** (cómo hacerlo rápido).